In [1]:
import cv2
import torch
import numpy as np
import onnxruntime as ort
import ipywidgets as widgets
import time
from IPython.display import display
from jetcam.utils import bgr8_to_jpeg
from jetracer.nvidia_racecar import NvidiaRacecar
from jetcam.csi_camera import CSICamera
from torch2trt import TRTModule
from utils import preprocess

# 1. Hardware
car = NvidiaRacecar()
camera = CSICamera(width=224, height=224, capture_fps=30)
camera.running = True

# 2. Road Following Model (TensorRT)
model_road = TRTModule()
model_road.load_state_dict(torch.load('rfm_red250_50e_loss_norbert_trt.pth'))

# 3. YOLO Model (ONNX)
providers = [
    ('TensorrtExecutionProvider', {'device_id': 0, 'trt_fp16_enable': True}),
    'CUDAExecutionProvider'
]
yolo_session = ort.InferenceSession('yolov4_1_3_224_224_dwapacholky.onnx', providers=providers)
yolo_input_name = yolo_session.get_inputs()[0].name

def road_inference(frame):
    img = preprocess(frame).half()
    output = model_road(img).detach().cpu().numpy().flatten()
    return float(output[0])

print("Inicjalizacja zakończona ✔")

Inicjalizacja zakończona ✔


In [2]:
image_widget = widgets.Image(format='jpeg', width=224, height=224)
display(image_widget)

Image(value=b'', format='jpeg', height='224', width='224')

In [7]:
# ====== PARAMETRY JAZDY ======
SPEED           = 0.3
K_LANE          = 1.9
AVOID_GAIN      = 0.8      
DANGER_Y        = 0.1      # 0.25 było zbyt wysoko na obrazie (za daleko)
AREA_THRESHOLD  = 0.020    # Obniżone, by łatwiej "zaskoczyło"
DEAD_ZONE       = 0       # Mniejszy dead_zone = szybsza reakcja
STEERING_BIAS   = 0.03
CONF_LIMIT      = 0.65      # 0.90 to za dużo dla Jetsona w ruchu
AVOID_HOLD_TIME = 0.2      # 1.5s to za długo, auto nie wróci na tor      # Podtrzymanie uniku w sekundach    
AVOID_DECAY     = 0.2

# --- PARAMETRY STOPU I KLAS ---
CLASS_AVOID       = 0         # Indeks klasy pachołka do omijania (pomarańczowy)
CLASS_STOP        = 1         # Indeks klasy czarnego pachołka
STOP_AREA_LIMIT   = 0.04      # Minimalna wielkość pachołka do stopu
STOP_COLLISION_PX = 35        # Szerokość korytarza stopu wokół linii RFM
RESUME_DELAY      = 2.0       # Ile sekund czekać po zabraniu pachołka STOP


print(f"Parametry załadowane! Próg wielkości (AREA): {AREA_THRESHOLD}")

Parametry załadowane! Próg wielkości (AREA): 0.02


In [10]:
# --- PARAMETRY KONFIGURACYJNE ---
MAX_SPEED = 0.3
MIN_SPEED = 0.3
STOP_DELAY = 2.0
STOP_RFM_ZONE = 50
STOP_MIN_AREA = 0.02
MAX_DETECTIONS = 5      # Ile pacholków max bierzemy pod uwagę na raz
# --------------------------------

active_avoidance = 0
stop_active = False
stop_lost_time = None

try:
    while True:
        frame = camera.value
        if frame is None: continue
        
        # 1. Obliczamy kierunek drogi (RFM)
        rfm_steer = road_inference(frame)
        lane_center_px = 112 + (rfm_steer * 112 * K_LANE)

        # 2. Detekcja przeszkód (YOLO)
        img_yolo = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB).transpose((2, 0, 1)).astype(np.float32) / 255.0
        blob = np.expand_dims(img_yolo, axis=0)
        yolo_outs = yolo_session.run(None, {yolo_input_name: blob})
        
        boxes = np.squeeze(yolo_outs[0])
        scores = np.squeeze(yolo_outs[1])

        # --- Obsługa dwóch klas ---
        if len(scores.shape) > 1:
            class_ids = np.argmax(scores, axis=1)
            confs     = np.max(scores, axis=1)
        else:
            class_ids = np.zeros(len(scores), dtype=int)
            confs     = scores

        current_frame_avoidance = 0
        best_conf  = 0
        best_area  = 0
        p_stop_detected  = False
        det_count        = 0   # liczba wykrytych pacholków powyżej progu

        if len(confs) > 0:
            # --- Bierzemy top-N detekcji powyżej progu, posortowanych po conf ---
            above_thresh = np.where(confs > CONF_LIMIT)[0]
            above_thresh = above_thresh[np.argsort(confs[above_thresh])[::-1]]  # malejąco
            above_thresh = above_thresh[:MAX_DETECTIONS]                         # max N
            det_count    = len(above_thresh)

            best_avoid_weight = 0  # zapamiętujemy najgroźniejszy avoid

            for idx in above_thresh:
                box = boxes[idx]
                x1, y1, x2, y2 = (box * 224).astype(int)
                area      = (box[2] - box[0]) * (box[3] - box[1])
                cx        = (x1 + x2) / 2
                y_bottom  = y2 / 224.0
                det_class = class_ids[idx]
                conf      = confs[idx]

                # Zapamiętaj dane najlepszej detekcji (dla overlay)
                if conf > best_conf:
                    best_conf = conf
                    best_area = area

                if det_class == 1:
                    # ── p_stop: sprawdź area i pozycję względem linii RFM ──
                    in_rfm_zone = abs(cx - lane_center_px) <= STOP_RFM_ZONE
                    big_enough  = area >= STOP_MIN_AREA

                    if in_rfm_zone and big_enough:
                        p_stop_detected = True
                        stop_lost_time  = None
                        stop_active     = True

                    color = (0, 0, 255)
                    label = f"STOP C:{conf:.2f} A:{area:.3f}"

                else:
                    # ── p_avoid: zbieramy najgroźniejszy sygnał ze wszystkich pacholków ──
                    if y_bottom > DANGER_Y:
                        relative_dist = cx - lane_center_px
                        if abs(relative_dist) > DEAD_ZONE:
                            weight = (y_bottom - DANGER_Y) / (1.0 - DANGER_Y)
                            avoid_signal = -AVOID_GAIN * weight if relative_dist > 0 else AVOID_GAIN * weight

                            # Bierzemy sygnał o największej wartości bezwzględnej
                            if abs(avoid_signal) > abs(best_avoid_weight):
                                best_avoid_weight      = avoid_signal
                                current_frame_avoidance = avoid_signal

                    color = (0, 255, 0)
                    label = f"AVOID C:{conf:.2f} A:{area:.3f}"

                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                cv2.putText(frame, label, (x1, y1-15),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1)

        # --- LOGIKA DELAY po zniknięciu p_stop ---
        if stop_active and not p_stop_detected:
            if stop_lost_time is None:
                stop_lost_time = time.time()
            elif time.time() - stop_lost_time >= STOP_DELAY:
                stop_active    = False
                stop_lost_time = None

        # --- LOGIKA DECAY ---
        if abs(current_frame_avoidance) > abs(active_avoidance):
            active_avoidance = current_frame_avoidance
        else:
            if active_avoidance > 0:
                active_avoidance = max(0, active_avoidance - AVOID_DECAY)
            elif active_avoidance < 0:
                active_avoidance = min(0, active_avoidance + AVOID_DECAY)

        # 3. Finalne sterowanie
        final_steering = np.clip(rfm_steer + active_avoidance + STEERING_BIAS, -1.0, 1.0)

        steering_impact = abs(final_steering)
        dynamic_speed   = MAX_SPEED - (steering_impact * (MAX_SPEED - MIN_SPEED))
        dynamic_speed   = np.clip(dynamic_speed, MIN_SPEED, MAX_SPEED)

        if stop_active:
            car.steering = STEERING_BIAS
            car.throttle = 0.0
            dynamic_speed = 0.0
        else:
            car.steering = -final_steering
            car.throttle = -dynamic_speed

        # Czas pozostały do ruszenia (dla debug)
        if stop_active and stop_lost_time is not None:
            remaining = max(0.0, STOP_DELAY - (time.time() - stop_lost_time))
        else:
            remaining = 0.0

        # 4. Debug Video Overlay
        cv2.putText(frame, f"CONF: {best_conf:.2f}",        (5, 15),  1, 0.4, (255, 255, 255), 1)
        cv2.putText(frame, f"AREA: {best_area:.3f}",        (5, 30),  1, 0.4, (255, 255, 255), 1)
        cv2.putText(frame, f"SPD:  {dynamic_speed:.2f}",    (5, 45),  1, 0.4, (0, 255, 0),     1)
        cv2.putText(frame, f"AVD:  {active_avoidance:.2f}", (5, 60),  1, 0.4, (0, 255, 255),   1)
        cv2.putText(frame, f"STP:  {int(stop_active)}",     (5, 75),  1, 0.4, (0, 0, 255),     1)
        cv2.putText(frame, f"DLY:  {remaining:.1f}s",       (5, 90),  1, 0.4, (0, 128, 255),   1)
        cv2.putText(frame, f"DET:  {det_count}",            (5, 105), 1, 0.4, (255, 255, 0),   1)

        cv2.line(frame, (int(lane_center_px), 0), (int(lane_center_px), 224), (0, 255, 0), 1)
        cv2.line(frame, (int(lane_center_px - STOP_RFM_ZONE), 0),
                        (int(lane_center_px - STOP_RFM_ZONE), 224), (0, 0, 255), 1)
        cv2.line(frame, (int(lane_center_px + STOP_RFM_ZONE), 0),
                        (int(lane_center_px + STOP_RFM_ZONE), 224), (0, 0, 255), 1)
        dy_px = int(DANGER_Y * 224)
        cv2.line(frame, (0, dy_px), (224, dy_px), (0, 255, 255), 2)

        image_widget.value = bgr8_to_jpeg(frame)
        print(f"\rSPD: {dynamic_speed:.2f} | ST: {-final_steering:.2f} | "
              f"Avoid: {active_avoidance:.2f} | Stop: {stop_active} | "
              f"Delay: {remaining:.1f}s | Det: {det_count}", end="")

except KeyboardInterrupt:
    print("\nZatrzymano.")
finally:
    car.throttle = 0.0
    car.steering = STEERING_BIAS

SPD: 0.30 | ST: 0.54 | Avoid: 0.00 | Stop: False | Delay: 0.0s | Det: 00
Zatrzymano.
